In [1]:
from diffwave.inference import predict as diffwave_predict
import os
import numpy as np
model_dir = 'diffwav_model/diffwave-ljspeech-22kHz-1000578.pt'
from diffwave.inference import *
from diffwave.preprocess import *
# audio is a GPU tensor in [N,T] format.

In [2]:
wm_path = './squarewav_test_50'
sound_file = os.listdir(wm_path)
save_pth = 'wavmark_square_diffrecover_50'
state = 'diff'
for i in sound_file:
    spectrogram = transform(f'{wm_path}/{i}')
    audio, sample_rate = diffwave_predict(torch.tensor(spectrogram,dtype=torch.float32).unsqueeze(0).cuda(), model_dir = model_dir, fast_sampling=True)
    torchaudio.save(f"./{save_pth}/{state}_{i.split('.')[0]}.wav", audio.cpu(), sample_rate)


/tmp/ipykernel_2724191/2063747114.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  audio, sample_rate = diffwave_predict(torch.tensor(spectrogram,dtype=torch.float32).unsqueeze(0).cuda(), model_dir = model_dir, fast_sampling=True)
/home/binhaoma/thinclient_drives/wavmark-main/diffwave/inference.py:35: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed

# Test Model

In [7]:
import wavmark
msg_path = './msg'
save_pth = 'wavmark_test_50'
save_dir = save_pth
test_data = os.listdir(save_dir)
model = wavmark.load_model().to('cuda')

/home/binhaoma/anaconda3/envs/SpeechGPT/lib/python3.8/site-packages/wavmark/__init__.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(resume_path

In [8]:
import soundfile
eer = []
for i in test_data:
    watermarked_signal, sample_rate = soundfile.read(f'{save_dir}/{i}')
    payload = torch.load(f"{msg_path}/{i.split('_')[-1]}_msg_tensor.pt")
    payload_decoded, _ = wavmark.decode_watermark(model, watermarked_signal, show_progress=True)
    BER = (np.array(payload) != payload_decoded).mean()
    result = round(BER, 5)
    eer.append(result)
1-sum(eer)/len(eer)

/tmp/ipykernel_2724191/2916973828.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  payload = torch.load(f"{msg_path}/{i.split('_')[-1]}_msg_tensor.pt")
100%|██████████| 2

1.0